# Introduction

This notebook showcases the capabilities and characteristics of the devised correlation disruption discovery approach.

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nedis.parallelization import set_threads_for_external_libraries
set_threads_for_external_libraries(4)

In [ ]:
import sys
import logging
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

import scipy
import sklearn

In [ ]:
import nedis

In [ ]:
from nedis.cluster.leidenalg import WeightedLeidenClustering
from nedis.cordis.disruption import CorrelationDisruption as CorrelationDisruptionTransformer
from nedis.data.synthetic import make_correlation_data_mixed
from nedis.visualization import plot_cordis_cluster

In [ ]:
from nedis.visualization import plot_cordis_cluster as plot_cluster

In [ ]:
from nedis.cordis.utils import calculate_disruption_values_for_cluster

In [ ]:
from nedis.base import calculate_correlation_disruption_matrix_cv

In [ ]:
logging.basicConfig(stream=sys.stdout)
logging.getLogger("nedis").setLevel(level=logging.DEBUG)

# Config

In [ ]:
# randomness
random_state = 43
np.random.seed(random_state)

In [ ]:
output_dir = pathlib.Path("../_out/synthetic")
output_dir.mkdir(parents=True, exist_ok=True)

# Heterosceasticity

In [ ]:
def format_ax(ax):
    # https://stackoverflow.com/a/27361819/991496

    # Hide the right and top spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    # Only show ticks on the left and bottom spines
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

In [ ]:
np.random.seed(13)

fig, axes = plt.subplots(2, 3, figsize=(3 * 1.2 * 3, 3 * 2), dpi=300)


# Changing correlation

X_r = np.random.multivariate_normal((0,0), [[1.0,0.9],[0.9,1.0]], size=50)
X_rr = np.random.multivariate_normal((0,0), [[1.0,0.4],[0.4,1.0]], size=50)

X = np.concatenate([X_r, X_rr]); X_idx = np.arange(X_r.shape[0])
disruption_values = np.mean(calculate_correlation_disruption_matrix_cv(X, idx_ref=X_idx), axis=(1,2))
disruption_ref = np.mean(disruption_values[:X_r.shape[0]])
disruption_samples = np.mean(disruption_values[X_r.shape[0]:])

ax = axes[0,0]
ax.scatter(X_r[:,0], X_r[:,1], label=f"reference")
ax.scatter(X_rr[:,0], X_rr[:,1], label=f"samples")
ax.legend()
ax.set(title="Correlation change", xlabel="feature 1", ylabel="feature 2")
format_ax(ax)

ax = axes[1,0]
format_ax(ax)
ax.set(ylabel="disruption")
sns.boxplot(x=np.repeat(["reference", "samples"], (X_r.shape[0], X_rr.shape[0])), y=disruption_values, ax=ax)

# Central tendency shift (feature 1)

X_r = np.random.multivariate_normal((0,0), [[1.0,0.9],[0.9,1.0]], size=50)
X_rr = X_r.copy(); X_rr[:,0] += 5

X = np.concatenate([X_r, X_rr]); X_idx = np.arange(X_r.shape[0])
disruption_values = np.mean(calculate_correlation_disruption_matrix_cv(X, idx_ref=X_idx), axis=(1,2))
disruption_ref = np.mean(disruption_values[:X_r.shape[0]])
disruption_samples = np.mean(disruption_values[X_r.shape[0]:])

ax = axes[0,1]
ax.scatter(X_r[:,0], X_r[:,1], label=f"reference")
ax.scatter(X_rr[:,0], X_rr[:,1], label=f"samples")
ax.legend()
ax.set(title="Shifting mean", xlabel="feature 1", ylabel="feature 2")
format_ax(ax)
                                                             
ax = axes[1,1]
format_ax(ax)
ax.set(ylabel="disruption")
sns.boxplot(x=np.repeat(["reference", "samples"], (X_r.shape[0], X_rr.shape[0])), y=disruption_values, ax=ax)

## Variance shift (feature 1)

X_r = np.random.multivariate_normal((0,0), [[1.0,0.9],[0.9,1.0]], size=50)
X_rr = X_r.copy(); X_rr[:,0] *= 5

X = np.concatenate([X_r, X_rr]); X_idx = np.arange(X_r.shape[0])
disruption_values = np.mean(calculate_correlation_disruption_matrix_cv(X, idx_ref=X_idx), axis=(1,2))
disruption_ref = np.mean(disruption_values[:X_r.shape[0]])
disruption_samples = np.mean(disruption_values[X_r.shape[0]:])

ax = axes[0,2]
ax.scatter(X_r[:,0], X_r[:,1], label=f"reference")
ax.scatter(X_rr[:,0], X_rr[:,1], label=f"samples")
ax.legend()
ax.set(title="Increasing variance", xlabel="feature 1", ylabel="feature 2")
format_ax(ax)
                                                                                                               
ax = axes[1,2]
format_ax(ax)
ax.set(ylabel="disruption")
sns.boxplot(x=np.repeat(["reference", "samples"], (X_r.shape[0], X_rr.shape[0])), y=disruption_values, ax=ax)

plt.tight_layout()
fig.savefig(output_dir / "heteroscedasticity.png", bbox_inches="tight")
fig.savefig(output_dir / "heteroscedasticity.pdf", bbox_inches="tight")

# Data

In [ ]:
n_samples = 100
correlations = np.linspace(0.1,.9, 5)
data = [
    make_correlation_data_mixed(
        [5,10,5,5], 
        correlations=np.array([
            [1-c,0,0,0],
            [0,c,0,0],
            [0,0,1-c,-(1-c)],
            [0,0,-(1-c),1-c]]), 
        n_noise_features=15, 
        n_samples=n_samples,
        random_state=random_state + i) 
    for i, c in enumerate(correlations)]

X = np.concatenate(data)
y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
entities = np.tile(np.arange(n_samples), len(correlations))

In [ ]:
y_unique = np.unique(y)
correlation_matrices = {}
for yy in y_unique: 
    cor = np.corrcoef(X[y == yy,:], rowvar=False)
    correlation_matrices[yy] = cor

In [ ]:
coordinates = {}
for yy, correlation_matrix in correlation_matrices.items(): 
    tsne = sklearn.manifold.TSNE(n_components=2, random_state=random_state, init="pca", learning_rate=200)
    coo = tsne.fit_transform(abs(correlation_matrix))
    coordinates[yy] = coo

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(6 * len(data), 4))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    cor = correlation_matrices[yy]
    nedis.visualization.visualize_feature_clusters(cor, ax=ax, heatmap_kwargs=dict(cmap="vlag", center=0, vmin=-1, vmax=1))

In [ ]:
fig, axes = plt.subplots(1, len(y_unique), figsize=(4 * 2 * len(y_unique), 4 * 2))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    ax.axis("off")
    ax.set_title(yy)
    plot_cordis_cluster(None, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)

# Sanity check: linear models

In [ ]:
# checking how well regular models will do

import sklearn.ensemble
# e = sklearn.linear_model.LinearRegression()  # this actually works well and is a lot better than ridge (or lasso or EN), surprisingly
# e = sklearn.linear_model.Ridge()
e = sklearn.ensemble.RandomForestRegressor()
# e = sklearn.svm.SVR()

e.fit(X, y)
print(scipy.stats.spearmanr(y, e.predict(X)))

e.fit(X[:,0:5], y)
print(scipy.stats.spearmanr(y, e.predict(X[:,0:5])))

e.fit(X[:,5:15], y)
print(scipy.stats.spearmanr(y, e.predict(X[:,5:15])))

e.fit(X[:,15:25], y)
print(scipy.stats.spearmanr(y, e.predict(X[:,15:25])))

e.fit(X[:,25:], y)
print(scipy.stats.spearmanr(y, e.predict(X[:,25:])))

# Sets of variables

In [ ]:
from nedis.cordis.clustering import ReferenceCorrelationMatrixClusteringStep
from nedis.cordis.optimization import GreedyRefinementOptimizationStep

In [ ]:
from nedis.cordis.utils import calculate_disruption_values_for_cluster_from_data

In [ ]:
clustering_algorithm = WeightedLeidenClustering(random_state=random_state)

clustering_step = ReferenceCorrelationMatrixClusteringStep(
    clustering_algorithm=clustering_algorithm,
#     clustering_absolute_correlation=True,
#     correlation_function="spearman",
#     feature_filters=None
)

optimization_step = GreedyRefinementOptimizationStep(
    separation_score="spearman",
#     separation_score_comparison='all',
#     refinement_mode="rows-and-columns",
#     correlation_function="spearman",
#     disruption_metric="direction",
#     disruption_robustness='loo',
#     disruption_aggregation='mean',
#     max_runs=-1
)

t = CorrelationDisruptionTransformer(
    clustering_step=clustering_step,
    cluster_optimization_step=optimization_step,
    filter_coverage_threshold=0.5, 
    separation_score_threshold=("auto", 1)
)
t.fit(X, y, subset_masks="y")

In [ ]:
t.filter_reset()
t.filter_clusters_by_threshold(separation_score_threshold=('auto', -1))
t.filter_clusters_by_overlap(filter_coverage_threshold=0.4)

In [ ]:
# t.clusters_

In [ ]:
clusters = list(sorted([c for c in t.clusters_ if c["selected"]], key=lambda c: c["reference_score"]))
len(clusters)

In [ ]:
if "disruption_matrices_dict" not in globals():
    disruption_matrices_dict = {}
    for yy in np.unique(y):
        print("Calculating disruption values for reference:", yy)
        disruption_matrices_dict[yy] = calculate_correlation_disruption_matrix_cv(X, idx_ref=(y == yy))
disruption_matrices = disruption_matrices_dict[0]

In [ ]:
cluster = {
    "reference_data": 0, 
    "reference_id": "overall", 
    "rows": np.arange(X.shape[1]), 
    "columns": np.arange(X.shape[1]),
    "edges": np.ones((X.shape[1], X.shape[1]))
}
# values = t.calculate_cluster_disruption_values(cluster, X, y)
values, _ = calculate_disruption_values_for_cluster(cluster, disruption_matrices, "mean")

In [ ]:
r, p = scipy.stats.spearmanr(values, y)

In [ ]:
# fig, axes = plt.subplots(1, len(y_unique), figsize=(4 * 2 * len(y_unique), 4 * 2))
# for i, yy in enumerate(y_unique):
#     ax = axes[i]
#     ax.axis("off")
#     ax.set_title(yy)
#     plot_cordis_cluster(None, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)

fig, axes = plt.subplots(1, len(y_unique) + 1, figsize=(4 * 1 * (len(y_unique) + 1), 4 * 1), dpi=300)

#     ax = axes[0]
#     sns.kdeplot(x=values, hue=y, ax=ax, palette=sns.color_palette(n_colors=len(y_unique)))
#     ax.set_title(f"{scipy.stats.spearmanr(values, y)[0]:.04f}")         

ax = axes[0]
x_rank = scipy.stats.rankdata(y, method="dense")
sns.lineplot(x=x_rank, y=values, hue=entities, color="blue", alpha=0.1, ax=ax)
sns.lineplot(x=x_rank, y=values, ax=ax)
ax.set(
    title=f"r={r:.02f}",
    xlabel="timepoints",
    xticks=np.unique(x_rank),
    xticklabels= y_unique,
    ylabel="disruption"
)
ax.get_legend().remove()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

for i, yy in enumerate(y_unique):
    ax = axes[i + 1]
    ax.axis("off")
    plot_cordis_cluster(None, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)
fig.suptitle(f"Cluster: Reference data={cluster['reference_data']}, Id={cluster['reference_id']}")

fig.savefig(output_dir / "modules_overview.png", bbox_inches="tight")
fig.savefig(output_dir / "modules_overview.pdf", bbox_inches="tight")

In [ ]:
for i_cluster, cluster in enumerate(reversed(clusters)):
    
    fig, axes = plt.subplots(1, len(y_unique) + 1, figsize=(4 * 1 * (len(y_unique) + 1), 4 * 1), dpi=300)
 
    values, _ = calculate_disruption_values_for_cluster(
        cluster, disruption_matrices_dict[cluster["reference_label"]], disruption_aggregation="mean")
   
#     ax = axes[0]
#     sns.kdeplot(x=values, hue=y, ax=ax, palette=sns.color_palette(n_colors=len(y_unique)))
#     ax.set_title(f"{scipy.stats.spearmanr(values, y)[0]:.04f}")
      
    r, p = scipy.stats.spearmanr(values, y)
    
    ax = axes[0]
    x_rank = scipy.stats.rankdata(y, method="dense")
    sns.lineplot(x=x_rank, y=values, hue=entities, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=x_rank, y=values, ax=ax)
    ax.set(
        xlabel="timepoints",
        xticks=np.unique(x_rank),
        xticklabels= y_unique,
        ylabel="disruption",
        title=f"r={r:.02f}"
    )
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    for i, yy in enumerate(y_unique):
        ax = axes[i + 1]
        ax.axis("off")
        plot_cluster(cluster, coordinates[y_unique[0]], correlation_matrices[yy], correlation_threshold=0, verbose=1, ax=ax)
    fig.suptitle(f"Cluster: Reference data={cluster['reference_label']}, Id={cluster['id']}")
    
    fig.savefig(output_dir / f"modules_{i_cluster}.png", bbox_inches="tight")
    fig.savefig(output_dir / f"modules_{i_cluster}.pdf", bbox_inches="tight")
    
    plt.show()
#     break

In [ ]:
for c in reversed(clusters):

    cluster = c  # clusters[-4]

    clustering = []
    for i in np.arange(X.shape[1]):
        if i in cluster["edges"].nonzero()[0] or i in cluster["edges"].nonzero()[1]:
            clustering.append(0)
        elif i in cluster["optimization"]["log"][0]["edges"].nonzero()[0] or i in cluster["optimization"]["log"][0]["edges"].nonzero()[1]:
            clustering.append(1)
        else:
            clustering.append(2)
    clustering = np.array(clustering)

    fig, axes = plt.subplots(1, len(y_unique), figsize=(12 * 2 * len(y_unique), 8 * 2))
    for i, yy in enumerate(y_unique):
        ax = axes[i]
#         ax.set_title(cluster["scores"][i])
        nedis.visualization.visualize_feature_clusters(
            correlation_matrices[i], 
            clustering=clustering, 
            axline_kwargs=dict(color="black"),
            heatmap_kwargs=dict(cmap="vlag", center=0, vmin=-1, vmax=1), 
            ax=ax)
    fig.suptitle(f"Cluster: Reference data={c['reference_label']}, Id={c['id']}")

In [ ]:
for i_cluster, c in enumerate(reversed(clusters)):

    g = sns.clustermap(correlation_matrices[0][c["rows"],:][:,c["columns"]])
    idx = g.dendrogram_row.reordered_ind
    plt.close()

    fig, axes = plt.subplots(1, len(y_unique), figsize=(4 * 2 * len(y_unique), 3 * 2))
    fig.suptitle(f"Cluster {i_cluster}: Reference data={c['reference_label']}, Id={c['id']}")
    for i, yy in enumerate(y_unique):
        ax = axes[i]
#         ax.set_title(cluster["scores"][i])
        sns.heatmap(
            correlation_matrices[i][c["rows"],:][:,c["columns"]][idx,:][:,idx], 
            vmin=-1, vmax=1, center=0,
            cmap="vlag",
            ax=ax)

In [ ]:
max_n_clusters = 5
values_topk = np.array([
    calculate_disruption_values_for_cluster(clusters[-i], disruption_matrices_dict[clusters[-i]["reference_label"]], disruption_aggregation="mean")[0] 
    for i in range(min(len(clusters), max_n_clusters))]).transpose()
values_topk.shape

In [ ]:
# looks different than before because we are calculating the mean now not the sum ... at least that is what I think
for i, (c, v) in enumerate(zip(reversed(clusters), reversed(values_topk.transpose()))):
    sns.lineplot(
        x=y, y=v,
        label=f"Cluster {i}: Reference data={c['reference_label']}, Id={c['id']}, Size={len(c['rows'])}/{len(c['columns'])}")

In [ ]:
fig, axes = plt.subplots(1, len(clusters), figsize=(4 * len(clusters), 4), sharex=True)
for i, (c, v) in enumerate(zip(reversed(clusters), reversed(values_topk.transpose()))):
    ax=axes[i]
    ax.set_title(f"Cluster {i}: \n Reference data={c['reference_label']}, \nId={c['id']}, \nSize={len(c['rows'])}/{len(c['columns'])}")
    sns.kdeplot(
        x=v,
        hue=y, 
        ax=ax)

# Comparison to dimensionality reduction

In [ ]:
for module_size in [2,4,8,16,32]:
    
    print("module size:", module_size)
    
    print("create data")
    n_samples = 100
    correlations = np.linspace(0.1,0.9, 5)
    data = [
        make_correlation_data_mixed(
            [module_size, 50 - module_size], 
            correlations=np.array([
                [1-c,0],
                [0,0]
            ]),
            n_noise_features=15, n_samples=n_samples, random_state=random_state + i) 
        for i, c in enumerate(correlations)]

    X = np.concatenate(data)
    y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
    entities = np.tile(np.arange(n_samples), len(correlations))

    print("calculate correlation matrices")
    y_unique = np.unique(y)
    correlation_matrices = {}
    for yy in y_unique: 
        cor = np.corrcoef(X[y == yy,:], rowvar=False)
        correlation_matrices[yy] = cor

    print("plot")
    fig, axes = plt.subplots(1, len(y_unique) + 1, figsize=(4 * 1 * (len(y_unique) + 1), 4 * 1), dpi=300)

    cluster = {"reference_label": 0, "id": "overall", "rows": np.arange(module_size), "columns": np.arange(module_size), "edges": np.ones((module_size, module_size))}
    values, _ = calculate_disruption_values_for_cluster_from_data(cluster, X, idx_ref=(y ==  cluster["reference_label"]))
    r, p = scipy.stats.spearmanr(y, values)
#     auc = sklearn.metrics.roc_auc_score(y, values, multi_class="ovr")

    ax = axes[0]
    x_rank = scipy.stats.rankdata(y, method="dense")
    sns.lineplot(x=x_rank, y=values, hue=entities, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=x_rank, y=values, ax=ax)
    ax.set(
        xlabel="timepoints",
        xticks=np.unique(x_rank),
        xticklabels= y_unique,
        ylabel="disruption"
    )
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set(title=f"n={module_size}, r={r:.02f}")
    ax.set_ylim([-0.01,0.006])

    for i, yy in enumerate(y_unique):
        ax = axes[i + 1]
        ax.axis("off")
        plot_cordis_cluster(cluster, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)
    fig.suptitle(f"Cluster: Reference data={cluster['reference_label']}, Id={cluster['id']}")
    
    fig.savefig(output_dir / f"module-size_{module_size:02d}.png", bbox_inches="tight")
    fig.savefig(output_dir / f"module-size_{module_size:02d}.pdf", bbox_inches="tight")
    
    fig.show()
    plt.show()


In [ ]:
print("plot")

cluster = {"reference_label": 0, "id": "overall", "rows": np.arange(module_size), "columns": np.arange(module_size), "edges": np.ones((module_size, module_size))}
values, _ = calculate_disruption_values_for_cluster_from_data(cluster, X, idx_ref=(y ==  cluster["reference_label"]))
r, p = scipy.stats.spearmanr(y, values)
#     auc = sklearn.metrics.roc_auc_score(y, values, multi_class="ovr")


fig, ax = plt.subplots(figsize=(4,3), dpi=300)
x_rank = scipy.stats.rankdata(y, method="dense")
sns.lineplot(x=x_rank, y=values, hue=entities, color="blue", alpha=0.1, ax=ax)
sns.lineplot(x=x_rank, y=values, ax=ax)
ax.set(
    xlabel="timepoints",
    xticks=np.unique(x_rank),
    xticklabels= y_unique,
    ylabel="disruption"
)
ax.get_legend().remove()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set(title=f"r={r:.02f}")
ax.set_ylim([-0.01,0.006])

fig.savefig(output_dir / f"dimensionality-reduction_module-size_{module_size:02d}.png", bbox_inches="tight")
fig.savefig(output_dir / f"dimensionality-reduction_module-size_{module_size:02d}.pdf", bbox_inches="tight")

fig.show()
plt.show()


In [ ]:
import sklearn.decomposition

X_ref = X[y == 0]
X_ref.shape

pca = sklearn.decomposition.PCA(random_state=random_state)
pca.fit(X_ref)

transformed = pca.transform(X)

for i in range(2):
    r, p = scipy.stats.spearmanr(y, transformed[:,i])
    
    fig, ax = plt.subplots(figsize=(4,3), dpi=300)
    sns.lineplot(x=y, y=transformed[:,i], hue=entities, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=y, y=transformed[:,i], ax=ax)
    
    ax.set(title=f"r={r:.02f}", xlabel="timepoint", ylabel="PCA")
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    fig.savefig(output_dir / f"dimensionality-reduction_pca_{i}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
import sklearn.manifold

tsne = sklearn.manifold.TSNE(n_components=1, random_state=random_state + 3)
transformed = tsne.fit_transform(X)


for i in range(1):
    r, p = scipy.stats.spearmanr(y, transformed[:,i])
    
    fig, ax = plt.subplots(figsize=(4,3), dpi=300)
    sns.lineplot(x=y, y=transformed[:,i], hue=entities, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=y, y=transformed[:,i], ax=ax)
    
    ax.set(title=f"r={r:.02f}", xlabel="timepoint", ylabel="TSNE")
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    fig.savefig(output_dir / f"dimensionality-reduction_tsne_{i}.pdf", bbox_inches="tight")
    plt.show()

# Increasing sets of variables

In [ ]:
n_samples = 100
correlations = np.linspace(0,.99, 5)
data = [
    make_correlation_data_mixed(
        [0,50], 
        correlations=np.array([
            [1-c,0],
            [0,0]
        ]),
        n_noise_features=15, 
        n_samples=n_samples,
        random_state=random_state + i) 
    for i, c in enumerate(correlations)]

X = np.concatenate(data)
y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
entities = np.tile(np.arange(n_samples), len(correlations))


In [ ]:
y_unique = np.unique(y)
correlation_matrices = {}
for yy in y_unique: 
    cor = np.corrcoef(X[y == yy,:], rowvar=False)
    correlation_matrices[yy] = cor

In [ ]:
coordinates = {}
for yy, correlation_matrix in correlation_matrices.items(): 
    tsne = sklearn.manifold.TSNE(n_components=2, random_state=random_state)
    coo = tsne.fit_transform(abs(correlation_matrix))
    coordinates[yy] = coo

In [ ]:
for module_size in [2,4,8,16,32]:
    
    print("module size:", module_size)
    
    print("create data")
    n_samples = 100
    correlations = np.linspace(0.1,0.9, 5)
    data = [
        make_correlation_data_mixed(
            [module_size, 50 - module_size], 
            correlations=np.array([
                [1-c,0],
                [0,0]
            ]),
            n_noise_features=15, n_samples=n_samples, random_state=random_state + i) 
        for i, c in enumerate(correlations)]

    X = np.concatenate(data)
    y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
    entities = np.tile(np.arange(n_samples), len(correlations))

    print("calculate correlation matrices")
    y_unique = np.unique(y)
    correlation_matrices = {}
    for yy in y_unique: 
        cor = np.corrcoef(X[y == yy,:], rowvar=False)
        correlation_matrices[yy] = cor

    print("plot")
    fig, axes = plt.subplots(1, len(y_unique) + 1, figsize=(4 * 1 * (len(y_unique) + 1), 4 * 1), dpi=300)

    cluster = {"reference_label": 0, "id": "overall", "rows": np.arange(module_size), "columns": np.arange(module_size), "edges": np.ones((module_size, module_size))}
    values, _ = calculate_disruption_values_for_cluster_from_data(cluster, X, idx_ref=(y ==  cluster["reference_label"]))
    r, p = scipy.stats.spearmanr(y, values)
#     auc = sklearn.metrics.roc_auc_score(y, values, multi_class="ovr")

    ax = axes[0]
    x_rank = scipy.stats.rankdata(y, method="dense")
    sns.lineplot(x=x_rank, y=values, hue=entities, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=x_rank, y=values, ax=ax)
    ax.set(
        xlabel="timepoints",
        xticks=np.unique(x_rank),
        xticklabels= y_unique,
        ylabel="disruption"
    )
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set(title=f"n={module_size}, r={r:.02f}")
    ax.set_ylim([-0.01,0.006])

    for i, yy in enumerate(y_unique):
        ax = axes[i + 1]
        ax.axis("off")
        plot_cordis_cluster(cluster, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)
    fig.suptitle(f"Cluster: Reference data={cluster['reference_label']}, Id={cluster['id']}")
    
    fig.savefig(output_dir / f"module-size_{module_size:02d}.png", bbox_inches="tight")
    fig.savefig(output_dir / f"module-size_{module_size:02d}.pdf", bbox_inches="tight")
    
    fig.show()
    plt.show()


# Increasing sets of samples

In [ ]:
n_samples = 100
correlations = np.linspace(0,.99, 5)
data = [
    make_correlation_data_mixed(
        [0,50], 
        correlations=np.array([
            [1-c,0],
            [0,0]
        ]),
        n_noise_features=15, n_samples=n_samples, random_state=random_state + i) 
    for i, c in enumerate(correlations)]

X = np.concatenate(data)
y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
entities = np.tile(np.arange(n_samples), len(correlations))

In [ ]:
y_unique = np.unique(y)
correlation_matrices = {}
for yy in y_unique: 
    cor = np.corrcoef(X[y == yy,:], rowvar=False)
    correlation_matrices[yy] = cor

In [ ]:
# use same as above
coordinates = {}
random_state = 42
for yy, correlation_matrix in correlation_matrices.items(): 
    tsne = sklearn.manifold.TSNE(n_components=2, random_state=random_state)
    coo = tsne.fit_transform(abs(correlation_matrix))
    coordinates[yy] = coo

In [ ]:
fig, axes = plt.subplots(1, len(y_unique) + 1, figsize=(4 * 1 * (len(y_unique) + 1), 4 * 1), dpi=300)

cluster = {"reference_label": 0, "id": "overall", "rows": np.arange(X.shape[1]), "columns": np.arange(X.shape[1]), "edges": np.ones((X.shape[1],X.shape[1]))}
values, _ = calculate_disruption_values_for_cluster_from_data(cluster, X,  y == cluster["reference_label"])

ax = axes[0]
x_rank = scipy.stats.rankdata(y, method="dense")
sns.lineplot(x=x_rank, y=values, hue=entities, color="blue", alpha=0.1, ax=ax)
sns.lineplot(x=x_rank, y=values, ax=ax)
ax.set(
    xlabel="timepoints",
    xticks=np.unique(x_rank),
    xticklabels= y_unique,
    ylabel="disruption"
)
ax.get_legend().remove()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

for i, yy in enumerate(y_unique):
    ax = axes[i + 1]
    ax.axis("off")
    plot_cordis_cluster(cluster, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)
fig.suptitle(f"Cluster: Reference data={cluster['reference_label']}, Id={cluster['id']}")


In [ ]:
# repeated disruption - average of individual samples

import pandas as pd

n_repeats = [1,2,4,8,16]


fig, axes = plt.subplots(1, len(n_repeats), figsize=(4 * 1 * (len(n_repeats)), 4 * 1), dpi=300, sharey=True)

for i, n_repeat in enumerate(n_repeats):
    
    print("repeat:", n_repeat)
    
    print("create data")
    module_size = 10
    n_entities = 100
    n_samples = n_entities * n_repeat
    
    correlations = np.linspace(0.2,.8, 5)
    data = [
        make_correlation_data_mixed(
            [module_size, 50 - module_size], 
            correlations=np.array([
                [1-c,0],
                [0,  0]
            ]),
            n_noise_features=15, n_samples=n_samples, random_state=random_state + i) 
        for i, c in enumerate(correlations)]

    X = np.concatenate(data)
    y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
    entities = np.tile(np.repeat(np.arange(n_entities), n_repeat), len(correlations))

    print("calculate disruptions")
    cluster = {"reference_label": 0, "id": "overall", "rows": np.arange(module_size), "columns": np.arange(module_size), "edges": np.zeros((X.shape[1],X.shape[1]))}
    cluster["edges"][0:module_size,0:module_size] = 1
#     cluster = {"reference_data": 0, "reference_id": "overall", "rows": np.arange(X.shape[1]), "columns": np.arange(X.shape[1])}
    values, _ = calculate_disruption_values_for_cluster_from_data(cluster, X, y == cluster["reference_label"])
    
    df = pd.DataFrame(dict(value=values, entity=entities, y=y))
    df = df.groupby(["entity", "y"]).mean()
    
    vv = df.reset_index()["value"].values
    yy = df.reset_index()["y"].values
    ee = df.reset_index()["entity"].values
    
    r, p = scipy.stats.spearmanr(yy, vv)
    print(r, p)
#     auc = sklearn.metrics.roc_auc_score(y, values, multi_class="ovr")

    print("plot")
    ax = axes[i]
    x_rank = scipy.stats.rankdata(yy, method="dense")
    sns.lineplot(x=x_rank, y=vv, hue=ee, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=x_rank, y=vv, ax=ax)
    ax.set(
        xlabel="timepoints",
        xticks=np.unique(x_rank),
        xticklabels= y_unique,
        ylabel="disruption"
    )
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set(title=f"repeated: {n_repeat}, r={r:.02f}")
    
fig.savefig(output_dir / f"sample-size_default.png", bbox_inches="tight")
fig.savefig(output_dir / f"sample-size_default.pdf", bbox_inches="tight")

In [ ]:
# repeated disruption - average of individual samples

import pandas as pd

n_repeats = [1,2,4,8,16]


fig, axes = plt.subplots(1, len(n_repeats), figsize=(4 * 1 * (len(n_repeats)), 4 * 1), dpi=300, sharey=True)

for i, n_repeat in enumerate(n_repeats):
    
    print("repeat:", n_repeat)
    
    print("create data")
    module_size = 10
    n_entities = 100
    n_samples = n_entities * n_repeat
    
    correlations = np.linspace(0.2,.8, 5)
    data = [
        make_correlation_data_mixed(
            [module_size, 50 - module_size], 
            correlations=np.array([
                [1-c,0],
                [0,  0]
            ]),
            n_noise_features=15, n_samples=n_samples, random_state=random_state + i) 
        for i, c in enumerate(correlations)]

    X = np.concatenate(data)
    y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
    entities = np.tile(np.repeat(np.arange(n_entities), n_repeat), len(correlations))
    samples = np.ravel_multi_index([entities, y], dims=(n_entities, len(correlations)))

    print("calculate disruptions")
    cluster = {"reference_label": 0, "id": "overall", "rows": np.arange(module_size), "columns": np.arange(module_size), "edges": np.zeros((X.shape[1],X.shape[1]))}
    cluster["edges"][0:module_size,0:module_size] = 1
    values, _ = calculate_disruption_values_for_cluster_from_data(cluster, X, y == cluster["reference_label"], samples=samples)
    value_entities, value_y = np.unravel_index(np.unique(samples), (n_entities, len(correlations)))
    
    df = pd.DataFrame(dict(value=values, entity=value_entities, y=value_y))
    df = df.groupby(["entity", "y"]).mean()
    
    vv = df.reset_index()["value"].values
    yy = df.reset_index()["y"].values
    ee = df.reset_index()["entity"].values
    
    r, p = scipy.stats.spearmanr(yy, vv)
    print(r, p)
#     auc = sklearn.metrics.roc_auc_score(y, values, multi_class="ovr")

    print("plot")
    ax = axes[i]
    x_rank = scipy.stats.rankdata(yy, method="dense")
    sns.lineplot(x=x_rank, y=vv, hue=ee, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=x_rank, y=vv, ax=ax)
    ax.set(
        xlabel="timepoints",
        xticks=np.unique(x_rank),
        xticklabels= y_unique,
        ylabel="disruption"
    )
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set(title=f"repeated: {n_repeat}, r={r:.02f}")
    
fig.savefig(output_dir / f"sample-size_grouped.png", bbox_inches="tight")
fig.savefig(output_dir / f"sample-size_grouped.pdf", bbox_inches="tight")

In [ ]:
print("Done")